
# Demo for Section 1 - io and preprocessing
# Steps:
# - reading csv with missing values
# - filling missing values, scaling data
# - one hot encoding text cols
# - making final matrix and timing check
# sample_data.csv should be in same folder



In [1]:
import sys, platform, pathlib
import numpy as np

# to import numcompute from demo folder
ROOT = pathlib.Path.cwd()
if not (ROOT / "numcompute").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from numcompute.io import load_csv, iter_csv_chunks
from numcompute.preprocessing import StandardScaler, MinMaxScaler, SimpleImputer, OneHotEncoder, NotFittedError

DATA = ROOT / "demo" / "sample_data.csv"
np.set_printoptions(precision=3, suppress=True)
print("NumPy", np.__version__, "| Python", platform.python_version())
print()

with open(DATA) as f:
    print(f.read())

NumPy 2.5.2 | Python 3.14.7

age,income,hours,city,plan
23,42000,38,Adelaide,basic
35,,40,Melbourne,premium
41,68000,45,Adelaide,premium
29,39000,NA,Sydney,basic
52,91000,50,Melbourne,
33,55000,37,Sydney,basic
47,,42,Adelaide,premium
38,73000,41,Brisbane,standard
26,36000,35,Sydney,basic
60,98000,48,Melbourne,premium
31,,39,,standard
44,82000,43,Adelaide,standard



# 1. Reading a CSV with missing values (io.py)The file has empty fields and the text NA. For the numeric columns we are reading, both of these become NaN. I used skip_header=1 to skip the header line, and usecols to pick only the numeric columns which are age, income and hours.

In [2]:
NUM = dict(skip_header=1, usecols=(0, 1, 2), missing_values="NA")
X = load_csv(DATA, **NUM)
print("shape:", X.shape, "| missing cells:", int(np.isnan(X).sum()))
print(X)

shape: (12, 3) | missing cells: 4
[[   23. 42000.    38.]
 [   35.    nan    40.]
 [   41. 68000.    45.]
 [   29. 39000.    nan]
 [   52. 91000.    50.]
 [   33. 55000.    37.]
 [   47.    nan    42.]
 [   38. 73000.    41.]
 [   26. 36000.    35.]
 [   60. 98000.    48.]
 [   31.    nan    39.]
 [   44. 82000.    43.]]


# nan_policy controls how to handle missing values. keep is the default  showed above, other options are omit or fill.

In [3]:
X_omit = load_csv(DATA, nan_policy="omit", **NUM)
X_fill = load_csv(DATA, nan_policy="fill", fill_value=-1, **NUM)
print("omit ->", X_omit.shape, "(rows containing a missing value are dropped)")
print("fill ->", X_fill.shape, "| NaN left:", int(np.isnan(X_fill).sum()))

omit -> (8, 3) (rows containing a missing value are dropped)
fill -> (12, 3) | NaN left: 0


For files too big for memory, iter_csv_chunks reads a few rows at a time. Stacking the blocks gives exactly the same result as load_csv.

In [4]:
chunks = list(iter_csv_chunks(DATA, 5, **NUM))
print("block shapes:", [c.shape for c in chunks])
print("same as load_csv:", np.array_equal(np.vstack(chunks), X, equal_nan=True))

block shapes: [(5, 3), (5, 3), (2, 3)]
same as load_csv: True


# Numeric preprocessing ('preprocessing.py')
# Every transformer uses fit, transform and fit_transform, and never changes the array that is passed in.
# First, the missing numbers are filled with the column mean.

In [5]:
imputer = SimpleImputer("mean").fit(X)
print("learned column means:", imputer.statistics_)
X_imp = imputer.transform(X)
print("NaN left:", int(np.isnan(X_imp).sum()))
print("original untouched (still has NaN):", bool(np.isnan(X).any()))

learned column means: [   38.25  64888.889    41.636]
NaN left: 0
original untouched (still has NaN): True


# StandardScaler gives each column a mean of 0 and a standard deviation of 1. MinMaxScaler squeezes each column into a chosen range.

In [6]:
scaler = StandardScaler().fit(X_imp)
Z = scaler.transform(X_imp)
print("standardised column means:", Z.mean(axis=0))
print("standardised column stds :", Z.std(axis=0))

M = MinMaxScaler(feature_range=(0, 1)).fit_transform(X_imp)
print("min-max column minimums:", M.min(axis=0), "| maximums:", M.max(axis=0))

standardised column means: [0. 0. 0.]
standardised column stds : [1. 1. 1.]
min-max column minimums: [0. 0. 0.] | maximums: [1. 1. 1.]


# The scalers can handle missing values on their own -- NaN is just ignored while figuring out the mean and spread, and it stays as NaN in the output.

In [7]:
Z_nan = StandardScaler().fit_transform(X)
print("NaN in input:", int(np.isnan(X).sum()), "| NaN in output:", int(np.isnan(Z_nan).sum()))

NaN in input: 4 | NaN in output: 4


# Text columns with OneHotEncoder
# City and plan are handled as text columns. If a value is missing there, it just gets labeled as unknown.

In [8]:
CAT = dict(skip_header=1, usecols=(3, 4), dtype=str, nan_policy="fill", fill_value="unknown")
C = load_csv(DATA, **CAT)
enc = OneHotEncoder().fit(C)
print("categories per column:", [c.tolist() for c in enc.categories_])

H = enc.transform(C)
print("shape:", C.shape, "->", H.shape)
print(H[:3])

categories per column: [['Adelaide', 'Brisbane', 'Melbourne', 'Sydney', 'unknown'], ['basic', 'premium', 'standard', 'unknown']]
shape: (12, 2) -> (12, 9)
[[1. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 1. 0. 0.]]


# if a category shows up that wasn't seen during fit, it normally throws an error. With handle_unknown="ignore", it just turns into a row of zeros instead.

In [9]:
new = np.array([["Perth", "basic"]])
try:
    enc.transform(new)
except ValueError as err:
    print("error:", err)
print("ignore ->", OneHotEncoder("ignore").fit(C).transform(new))

error: found unseen categories during transform: 'Perth' in feature 0; use handle_unknown='ignore' to encode them as zeros
ignore -> [[0. 0. 0. 0. 0. 1. 0. 0. 0.]]


# putting it together and how errors are handled

In [10]:
final = np.hstack([Z, H])
print("final feature matrix:", final.shape, "= 3 scaled numeric columns +", H.shape[1], "one-hot columns")

try:
    StandardScaler().transform(X_imp)
except NotFittedError as err:
    print(type(err).__name__, "->", err)
print("NotFittedError is a ValueError:", issubclass(NotFittedError, ValueError))

final feature matrix: (12, 12) = 3 scaled numeric columns + 9 one-hot columns
NotFittedError -> StandardScaler is not fitted yet; call fit(X) before transform(X)
NotFittedError is a ValueError: True


# Python loops vs vectorised code
# Same job, done two ways — once with plain Python loops and once with the vectorised numcompute classes. The random seed is fixed so the data stays the same each run, and timing is just the best out of a few runs. Times will look different on different computers anyway.

In [ ]:
import time

def best_of(fn, repeats=5):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)

rng = np.random.default_rng(0)
big = rng.normal(size=(20_000, 10))
cats = rng.integers(0, 20, size=(20_000, 5))

def standardize_loop(A):
    out = np.empty_like(A)
    for j in range(A.shape[1]):
        col = A[:, j]
        mean = 0.0
        for v in col:
            mean += v
        mean /= len(col)
        var = 0.0
        for v in col:
            var += (v - mean) ** 2
        std = (var / len(col)) ** 0.5
        for i in range(len(col)):
            out[i, j] = (col[i] - mean) / std
    return out

def onehot_loop(A):
    cat_lists = [np.unique(A[:, j]) for j in range(A.shape[1])]
    out = np.zeros((A.shape[0], sum(len(c) for c in cat_lists)))
    for i in range(A.shape[0]):
        offset = 0
        for j, c in enumerate(cat_lists):
            out[i, offset + int(np.searchsorted(c, A[i, j]))] = 1.0
            offset += len(c)
    return out

# the loop versions should give the same answer as the vectorised ones
assert np.allclose(standardize_loop(big), StandardScaler().fit_transform(big))
assert np.array_equal(onehot_loop(cats), OneHotEncoder().fit_transform(cats))

rows = [
    ("StandardScaler", best_of(lambda: standardize_loop(big), 3), best_of(lambda: StandardScaler().fit_transform(big))),
    ("OneHotEncoder", best_of(lambda: onehot_loop(cats), 3), best_of(lambda: OneHotEncoder().fit_transform(cats))),
]
print(f"{'component':<16}{'loop (s)':>10}{'vectorised (s)':>16}{'speed-up':>10}")
for name, t_loop, t_vec in rows:
    print(f"{name:<16}{t_loop:>10.4f}{t_vec:>16.4f}{t_loop / t_vec:>9.0f}x")
print()
print("Environment:", platform.platform(), "| Python", platform.python_version(), "| NumPy", np.__version__)

component         loop (s)  vectorised (s)  speed-up
StandardScaler      0.0698          0.0049       14x
OneHotEncoder       0.1211          0.0091       13x

Environment: Windows-11-10.0.26200-SP0 | Python 3.14.7 | NumPy 2.5.2
